# Inverse kinematics

[Joints and frames](joints_and_frames.ipynb) went from joint angles to the
hand's position. This module goes the other way: given a point in front of the
robot, it finds arm angles that put the right palm there, sends them, and
measures how close the real palm got.

The arm has 7 joints and a point has 3 coordinates, so there are many answers,
or none if the point is out of reach. The numerical solver here handles both.

Run on the mock first (`HOST = "mock"`).

In [ ]:
import os, time, math
import numpy as np
import matplotlib.pyplot as plt
import g1lab

# "mock" runs the robot on this laptop (needs `pip install mujoco`).
# In the lab, the robot computer: HOST = "192.168.123.222"
HOST = os.environ.get("G1LAB_HOST", "mock")
NAME = os.environ.get("G1LAB_NAME", "student")     # your name: it shows on the lab board

r = g1lab.mock(name=NAME) if HOST == "mock" else g1lab.connect(HOST, name=NAME)
info = r.info()
print("connected to", HOST, "| session:", info.phase, "| has the robot:", info.owner)

## 1. Forward kinematics

The `fk()` from [Joints and frames](joints_and_frames.ipynb), compacted.

In [ ]:
import xml.etree.ElementTree as ET

URDF = os.path.join(os.path.dirname(g1lab.__file__), "description", "g1_29dof_dx3_ros.urdf")
JOINT_OF_CHILD = {j.find("child").get("link"): j for j in ET.parse(URDF).getroot().findall("joint")}

def rot_axis(axis, q):
    k = np.asarray(axis, float) / np.linalg.norm(axis)
    K = np.array([[0, -k[2], k[1]], [k[2], 0, -k[0]], [-k[1], k[0], 0]])
    return np.eye(3) + math.sin(q) * K + (1 - math.cos(q)) * K @ K

def rot_rpy(roll, pitch, yaw):
    return rot_axis([0, 0, 1], yaw) @ rot_axis([0, 1, 0], pitch) @ rot_axis([1, 0, 0], roll)

def chain(link):
    out = []
    while link in JOINT_OF_CHILD:
        out.append(JOINT_OF_CHILD[link]); link = out[-1].find("parent").get("link")
    return out[::-1]

def fk(angles, link="right_hand_palm_link"):
    T = np.eye(4)
    for j in chain(link):
        o, step = j.find("origin"), np.eye(4)
        step[:3, :3] = rot_rpy(*map(float, o.get("rpy", "0 0 0").split()))
        step[:3, 3] = list(map(float, o.get("xyz", "0 0 0").split()))
        T = T @ step
        if j.get("type") == "revolute":
            turn = np.eye(4)
            turn[:3, :3] = rot_axis(list(map(float, j.find("axis").get("xyz").split())),
                                    angles.get(j.get("name"), 0.0))
            T = T @ turn
    return T

RIGHT = [f"right_{n}_joint" for n in g1lab.ARM]

def palm(q7):
    """The right palm's position (pelvis frame, m) for the 7 right-arm angles."""
    return fk(dict(zip(RIGHT, q7)))[:3, 3]

home = info.home["right_arm"].copy()
lo, hi = info.limits["right_arm"].T
print("palm at home:", np.round(palm(home), 3))

## 2. The Jacobian

The Jacobian *J* (3×7) says how the palm moves when each joint turns a
little: column *i* is ∂palm/∂q<sub>i</sub>. You can get it numerically by
turning one joint at a time by a small ε, which is what `jacobian(q7)` does.

In [ ]:
def jacobian(q7, eps=1e-6):
    p0 = palm(q7)
    J = np.zeros((3, 7))
    for i in range(7):
        dq = np.array(q7, float); dq[i] += eps
        J[:, i] = (palm(dq) - p0) / eps
    return J

In [ ]:
J = jacobian(home)
print(np.round(J, 3))
print("rank", np.linalg.matrix_rank(J), "| singular values", np.round(np.linalg.svd(J)[1], 3))

## 3. Solving: damped least squares

Each step moves the joints by

  Δq = Jᵀ (J Jᵀ + λ² I)⁻¹ · (target − palm(q))

and keeps them inside the joint limits. λ (damping) keeps the step small
near poses where the arm cannot move the palm in some direction. `ik(target,
q_start)` below caps the error it feeds in at 5 cm per step, because the
formula is only right for small moves. It stops when the palm is within 1 mm,
or gives up after 300 steps, and returns the angles and the distance left to
the target (m).

It also adds a small pull toward the home pose in J's null space, so the arm
stays natural: Δq += (I − J⁺J) · k · (home − q).

In [ ]:
def ik(target, q_start, damping=0.05, iters=300, tol=1e-3, pull=0.05, max_step=0.05):
    q = np.array(q_start, float)
    for _ in range(iters):
        err = np.asarray(target) - palm(q)
        if np.linalg.norm(err) < tol:
            break
        err = err * min(1.0, max_step / np.linalg.norm(err))   # small steps only
        J = jacobian(q)
        Jinv = J.T @ np.linalg.inv(J @ J.T + damping**2 * np.eye(3))
        dq = Jinv @ err + (np.eye(7) - Jinv @ J) @ (pull * (home - q))
        q = np.clip(q + dq, lo, hi)
    return q, float(np.linalg.norm(np.asarray(target) - palm(q)))

In [ ]:
TARGETS = {                                   # pelvis frame: x forward, y left, z up (m)
    "in front":         (0.30, -0.20, 0.05),
    "high and right":   (0.25, -0.35, 0.20),
    "across the body":  (0.30, -0.05, 0.10),
    "too far":          (0.80, -0.20, 0.10),
}
solutions = {}
for name, p in TARGETS.items():
    q, miss = ik(p, home)
    solutions[name] = q
    print(f"{name:16s} miss {1000 * miss:6.1f} mm   angles (deg) {np.round(np.degrees(q), 0)}")

For the point that is too far, the solver stretches the arm as far toward it as it can and reports how far off it still is, about 16 in here. Check that number before you send the pose.

## 4. On the robot

Send each answer and measure where the palm really went: read the joints,
put them through `fk()`. A pose your solver found can still be **refused**:
the robot checks every pose against its own body before it moves (see
[the guide](../docs/guide.md), "When something goes wrong").

In [ ]:
def measured_palm():
    js = r.state.joints()
    return palm(js.arm("right"))

with r.control() as c:
    for name, q in solutions.items():
        try:
            c.arms.move_to(right=q, duration=2.0)
        except g1lab.Refused as e:
            print(f"{name:16s} REFUSED: {e.reason}")
            continue
        time.sleep(0.7)                       # let it settle (the arm lags ~0.1 s)
        got = measured_palm()
        print(f"{name:16s} palm {np.round(got, 3)}  off the target by "
              f"{1000 * np.linalg.norm(got - TARGETS[name]):5.1f} mm"
              + (f"  ({c.status.held})" if c.status.held else ""))
    c.arms.home()

**A pose the robot refuses.** This target is low, just in front of the thigh.
The solver finds angles for it, the robot refuses them, and the program prints
the reason and carries on. More in
[Refusals and errors](refusals_and_errors.ipynb).

In [ ]:
# A target that would hit the body is refused.
q, miss = ik((0.10, -0.05, -0.10), home)
with r.control() as c:
    try:
        c.arms.move_to(right=q, duration=2.0)
        print("accepted: the palm is at", np.round(measured_palm(), 3))
    except g1lab.Refused as e:
        print("refused, as expected:", e.reason)

The millimetres left between a target and the measured palm come from the
arm's lag, the controller, and how exactly the model matches the real arm.

**Try this.** Add your own points to `TARGETS` and run parts 3 and 4 again. Which
ones are out of reach, and which does the robot refuse?